# Iris Dataset Exploration with Apache Spark

このNotebookでは、Apache Sparkを使用してIrisデータセットを探索し、機械学習モデルを構築します。

In [15]:
// Spark 依存関係の読み込み（Scala 2.13 を明示的に指定）
import $ivy.`org.apache.spark:spark-sql_2.13:3.5.0`
import $ivy.`org.apache.spark:spark-mllib_2.13:3.5.0`

println("Spark 依存関係が正常にロードされました")

Spark 依存関係が正常にロードされました


import $ivy.$
import $ivy.$

## 1. 環境設定とライブラリのインポート

In [16]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.classification.DecisionTreeClassifier
import org.apache.spark.ml.feature.{StringIndexer, VectorAssembler}
import org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator

// SparkSessionの作成
val spark = SparkSession.builder()
  .appName("IrisExploration")
  .master("local[*]")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

println("Spark Session created successfully!")
println(s"Spark version: ${spark.version}")

Spark Session created successfully!
Spark version: 3.5.0


import org.apache.spark.sql.SparkSession
import org.apache.spark.ml.{Pipeline, PipelineModel}
import org.apache.spark.ml.classification.DecisionTreeClassifier
import org.apache.spark.ml.feature.{StringIndexer, VectorAssembler}
import org.apache.spark.ml.evaluation.MulticlassClassificationEvaluator
spark: SparkSession = org.apache.spark.sql.SparkSession@20e98d0

## 2. データの読み込み

In [17]:
// データの読み込み
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("../data/iris.csv")

println(s"データ件数: ${df.count()}")
println("\nスキーマ:")
df.printSchema()

データ件数: 150

スキーマ:
root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)



df: org.apache.spark.sql.package.DataFrame = [sepal_length: double, sepal_width: double ... 3 more fields]

## 3. データの概要確認

In [18]:
// 最初の5行を表示
df.show(5, truncate = false)

+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|species    |
+------------+-----------+------------+-----------+-----------+
|0.22        |0.63       |0.08        |0.04       |Iris-setosa|
|0.17        |0.42       |0.35        |0.04       |Iris-setosa|
|0.11        |0.5        |0.13        |0.04       |Iris-setosa|
|0.08        |0.46       |0.26        |0.04       |Iris-setosa|
|0.19        |0.67       |0.44        |0.04       |Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows



In [19]:
// 統計情報
df.describe("sepal_length", "sepal_width", "petal_length", "petal_width").show()

+-------+-------------------+-------------------+-------------------+-------------------+
|summary|       sepal_length|        sepal_width|       petal_length|        petal_width|
+-------+-------------------+-------------------+-------------------+-------------------+
|  count|                148|                150|                150|                150|
|   mean|0.42087837837837844|0.44000000000000017| 0.4889999999999999| 0.4506666666666667|
| stddev| 0.2289102520091739|0.18059558438863108|0.23100524413699344|0.30921347867047916|
|    min|                0.0|                0.0|               0.01|               0.01|
|    max|               0.94|                1.0|               0.95|               0.96|
+-------+-------------------+-------------------+-------------------+-------------------+



In [20]:
// 種類ごとの件数
df.groupBy("species").count().show()

+---------------+-----+
|        species|count|
+---------------+-----+
| Iris-virginica|   50|
|    Iris-setosa|   50|
|Iris-versicolor|   50|
+---------------+-----+



## 4. 特徴量の準備

In [21]:
// ラベルのインデックス化
val labelIndexer = new StringIndexer()
  .setInputCol("species")
  .setOutputCol("label")

// 特徴量のベクトル化
val assembler = new VectorAssembler()
  .setInputCols(Array("sepal_length", "sepal_width", "petal_length", "petal_width"))
  .setOutputCol("features")
  .setHandleInvalid("skip")

// パイプラインで変換
val prepPipeline = new Pipeline().setStages(Array(labelIndexer, assembler))
val preparedDf = prepPipeline.fit(df).transform(df)

println(s"準備後のデータ件数: ${preparedDf.count()}")
preparedDf.select("features", "label", "species").show(5, truncate = false)

準備後のデータ件数: 148
+---------------------+-----+-----------+
|features             |label|species    |
+---------------------+-----+-----------+
|[0.22,0.63,0.08,0.04]|0.0  |Iris-setosa|
|[0.17,0.42,0.35,0.04]|0.0  |Iris-setosa|
|[0.11,0.5,0.13,0.04] |0.0  |Iris-setosa|
|[0.08,0.46,0.26,0.04]|0.0  |Iris-setosa|
|[0.19,0.67,0.44,0.04]|0.0  |Iris-setosa|
+---------------------+-----+-----------+
only showing top 5 rows



labelIndexer: StringIndexer = strIdx_65b13f11a981
assembler: VectorAssembler = VectorAssembler: uid=vecAssembler_4c86044896c4, handleInvalid=skip, numInputCols=4
prepPipeline: Pipeline = pipeline_258992d1e0b0
preparedDf: org.apache.spark.sql.package.DataFrame = [sepal_length: double, sepal_width: double ... 5 more fields]

## 5. データの分割

In [22]:
// 訓練データとテストデータに分割
val Array(trainData, testData) = preparedDf.randomSplit(Array(0.7, 0.3), seed = 42)

println(s"訓練データ: ${trainData.count()} 件")
println(s"テストデータ: ${testData.count()} 件")

訓練データ: 102 件
テストデータ: 46 件


trainData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [sepal_length: double, sepal_width: double ... 5 more fields]
testData: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [sepal_length: double, sepal_width: double ... 5 more fields]

## 6. モデルの訓練

In [23]:
// Decision Treeモデルの作成
val dt = new DecisionTreeClassifier()
  .setLabelCol("label")
  .setFeaturesCol("features")
  .setMaxDepth(5)

val pipeline = new Pipeline().setStages(Array(dt))

// モデルの訓練
println("モデルを訓練中...")
val model = pipeline.fit(trainData)
println("訓練完了！")

モデルを訓練中...
訓練完了！


dt: DecisionTreeClassifier = dtc_efddc37ce122
pipeline: Pipeline = pipeline_9ecf6d050da0
model: PipelineModel = pipeline_9ecf6d050da0

## 7. モデルの評価

In [24]:
// テストデータで予測
val predictions = model.transform(testData)

// 評価メトリクスの計算
val evaluator = new MulticlassClassificationEvaluator()
  .setLabelCol("label")
  .setPredictionCol("prediction")

val accuracy = evaluator.setMetricName("accuracy").evaluate(predictions)
val precision = evaluator.setMetricName("weightedPrecision").evaluate(predictions)
val recall = evaluator.setMetricName("weightedRecall").evaluate(predictions)
val f1 = evaluator.setMetricName("f1").evaluate(predictions)

println(f"Accuracy: ${accuracy * 100}%.2f%%")
println(f"Precision: ${precision * 100}%.2f%%")
println(f"Recall: ${recall * 100}%.2f%%")
println(f"F1 Score: ${f1 * 100}%.2f%%")

Accuracy: 97.83%
Precision: 98.04%
Recall: 97.83%
F1 Score: 97.85%


predictions: org.apache.spark.sql.package.DataFrame = [sepal_length: double, sepal_width: double ... 8 more fields]
evaluator: MulticlassClassificationEvaluator = MulticlassClassificationEvaluator: uid=mcEval_9bc1598d6291, metricName=f1, metricLabel=0.0, beta=1.0, eps=1.0E-15
accuracy: Double = 0.9782608695652174
precision: Double = 0.9804347826086957
recall: Double = 0.9782608695652173
f1: Double = 0.9784823208090352

## 8. 予測結果の確認

In [25]:
// 予測結果のサンプル表示
predictions.select(
  "sepal_length", "sepal_width", "petal_length", "petal_width",
  "species", "label", "prediction"
).show(10, truncate = false)

+------------+-----------+------------+-----------+---------------+-----+----------+
|sepal_length|sepal_width|petal_length|petal_width|species        |label|prediction|
+------------+-----------+------------+-----------+---------------+-----+----------+
|0.03        |0.38       |0.02        |0.04       |Iris-setosa    |0.0  |0.0       |
|0.08        |0.46       |0.26        |0.04       |Iris-setosa    |0.0  |0.0       |
|0.08        |0.58       |0.37        |0.08       |Iris-setosa    |0.0  |0.0       |
|0.08        |0.67       |0.32        |0.04       |Iris-setosa    |0.0  |0.0       |
|0.14        |0.42       |0.44        |0.08       |Iris-setosa    |0.0  |0.0       |
|0.14        |0.46       |0.15        |0.04       |Iris-setosa    |0.0  |0.0       |
|0.14        |0.58       |0.3         |0.04       |Iris-setosa    |0.0  |0.0       |
|0.17        |0.42       |0.35        |0.04       |Iris-setosa    |0.0  |0.0       |
|0.17        |0.46       |0.4         |0.02       |Iris-setosa   

In [26]:
// 予測の正誤を確認
val correct = predictions.filter("label == prediction").count()
val total = predictions.count()

println(s"正解: $correct / $total")
println(f"正解率: ${correct.toDouble / total * 100}%.2f%%")

正解: 45 / 46
正解率: 97.83%


correct: Long = 45L
total: Long = 46L

## 9. Decision Treeの構造確認

In [27]:
// Decision Treeの構造を表示
val dtModel = model.stages(0).asInstanceOf[org.apache.spark.ml.classification.DecisionTreeClassificationModel]

println("Decision Tree Structure:")
println(dtModel.toDebugString)

Decision Tree Structure:
DecisionTreeClassificationModel: uid=dtc_efddc37ce122, depth=5, numNodes=17, numClasses=3, numFeatures=4
  If (feature 3 <= 0.295)
   Predict: 0.0
  Else (feature 3 > 0.295)
   If (feature 3 <= 0.65)
    If (feature 2 <= 0.655)
     If (feature 0 <= 0.765)
      Predict: 1.0
     Else (feature 0 > 0.765)
      Predict: 2.0
    Else (feature 2 > 0.655)
     If (feature 0 <= 0.5700000000000001)
      If (feature 0 <= 0.515)
       Predict: 1.0
      Else (feature 0 > 0.515)
       Predict: 2.0
     Else (feature 0 > 0.5700000000000001)
      Predict: 1.0
   Else (feature 3 > 0.65)
    If (feature 2 <= 0.435)
     If (feature 1 <= 0.44)
      Predict: 2.0
     Else (feature 1 > 0.44)
      Predict: 1.0
    Else (feature 2 > 0.435)
     Predict: 2.0



dtModel: org.apache.spark.ml.classification.DecisionTreeClassificationModel = DecisionTreeClassificationModel: uid=dtc_efddc37ce122, depth=5, numNodes=17, numClasses=3, numFeatures=4

## 10. クリーンアップ

In [28]:
// SparkSessionの停止
// spark.stop()
println("完了！")

完了！
